# Anotação Manual e Calibração do Threshold

**Pré-requisito:** `01_exploration.ipynb` — normalização `selective` já adotada.

Escopo deste notebook:
1. Gerar candidatos de referência (top-5 por skill) para orientar as anotações.
2. Anotar manualmente ~50 pares `skill_raw → taxonomy_id` (ground truth real).
3. Comparar curvas Precision-Recall: GT manual vs. GT sintético.
4. Distribuição de scores: modelo final (threshold 0.45) vs. anterior (0.60).
5. Calibrar e justificar o threshold definitivo.

## 1. Setup

In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.insert(0, 'src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

from match import load_skills, load_taxonomy
from embedder import SkillEmbedder
from matcher import SkillMatcher
from evaluator import evaluate, precision_recall_curve_by_threshold

SKILLS_PATH      = 'data/raw/new_skills.csv'
TAXONOMY_PATH    = 'data/raw/skill_taxonomy.csv'
ANNOTATIONS_PATH = 'data/raw/manual_annotations.csv'

# threshold calibrado via GT manual neste notebook
THRESHOLD = 0.45

skills_df   = load_skills(SKILLS_PATH)
taxonomy_df = load_taxonomy(TAXONOMY_PATH)

# normalização seletiva já aplicada em SkillEmbedder.normalize()
# (definida e validada no notebook 01)
embedder = SkillEmbedder()
matcher  = SkillMatcher(embedder=embedder, threshold=THRESHOLD)

print(f'Skills: {len(skills_df)} | Taxonomia: {len(taxonomy_df)}')
print(f'Threshold: {THRESHOLD} | Normalização: selective')


## 2. Candidatos de referência (top-5 por skill)

In [9]:
TOP_N = 5

skill_embs    = embedder.embed(skills_df['skill_raw'].tolist())
taxonomy_embs = embedder.embed_taxonomy(taxonomy_df)
sim_matrix    = cosine_similarity(skill_embs, taxonomy_embs)

rows = []
for i, skill_raw in enumerate(skills_df['skill_raw']):
    top_idx = np.argsort(sim_matrix[i])[::-1][:TOP_N]
    for rank, idx in enumerate(top_idx, 1):
        rows.append({
            'skill_raw':     skill_raw,
            'rank':          rank,
            'taxonomy_id':   int(taxonomy_df.iloc[idx]['id']),
            'taxonomy_name': taxonomy_df.iloc[idx]['skill_name'],
            'score':         round(float(sim_matrix[i, idx]), 4),
        })

candidates_df = pd.DataFrame(rows)

# exibe agrupado por skill
for skill, grp in candidates_df.groupby('skill_raw', sort=False):
    print(f'\n>>> "{skill}"')
    print(grp[['rank','taxonomy_id','taxonomy_name','score']].to_string(index=False))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]


>>> "gestão de pessoas"
 rank  taxonomy_id          taxonomy_name  score
    1            1      Gestão de Pessoas 0.7346
    2           26 Gestão de Stakeholders 0.5583
    3           24               Mentoria 0.5066
    4           13            Colaboração 0.4777
    5           11             Autogestão 0.4737

>>> "People Management"
 rank  taxonomy_id          taxonomy_name  score
    1            1      Gestão de Pessoas 0.7621
    2           26 Gestão de Stakeholders 0.5601
    3           28     Gestão de Mudanças 0.4810
    4           24               Mentoria 0.4787
    5           13            Colaboração 0.4644

>>> "SQL"
 rank  taxonomy_id      taxonomy_name  score
    1           32                SQL 0.7246
    2           59            MongoDB 0.5368
    3           58         PostgreSQL 0.5313
    4           57 Qualidade de Dados 0.5071
    5           39   Análise de Dados 0.3960

>>> "sql avançado"
 rank  taxonomy_id      taxonomy_name  score
    1           

## 3. Anotação manual

In [15]:
ANNOTATIONS = [
    # --- skills do arquivo new_skills.csv ---
    {"skill_raw": "gestão de pessoas",       "true_taxonomy_id": 1},
    {"skill_raw": "People Management",        "true_taxonomy_id": 1},
    {"skill_raw": "SQL",                      "true_taxonomy_id": 32},
    {"skill_raw": "sql avançado",             "true_taxonomy_id": 32},
    {"skill_raw": "machine learning",         "true_taxonomy_id": 40},
    {"skill_raw": "comunicação assertiva",    "true_taxonomy_id": 2},
    {"skill_raw": "python",                   "true_taxonomy_id": 31},
    {"skill_raw": "PYTHON",                   "true_taxonomy_id": 31},
    {"skill_raw": "análise de dados",         "true_taxonomy_id": 39},
    {"skill_raw": "resolução de conflitos",   "true_taxonomy_id": 3},
    {"skill_raw": "power bi",                 "true_taxonomy_id": 76},
    {"skill_raw": "PowerBI",                  "true_taxonomy_id": 76},
    {"skill_raw": "espiritualidade",          "true_taxonomy_id": None},  # None
    {"skill_raw": "excel avançado",           "true_taxonomy_id": 78},
    {"skill_raw": "scrum",                    "true_taxonomy_id": 19},
    {"skill_raw": "Metodologias Ágeis",       "true_taxonomy_id": 19},
    {"skill_raw": "storytelling com dados",   "true_taxonomy_id": 21},
    {"skill_raw": "pensamento crítico",       "true_taxonomy_id": 6},
    {"skill_raw": "Critical Thinking",        "true_taxonomy_id": 6},
    {"skill_raw": "versionamento de código",  "true_taxonomy_id": 72},
    {"skill_raw": "Git",                      "true_taxonomy_id": 72},
    {"skill_raw": "cloud computing",          "true_taxonomy_id": [66, 67, 68]},
    {"skill_raw": "AWS",                      "true_taxonomy_id": 66},
    {"skill_raw": "engenharia de dados",      "true_taxonomy_id": 51},
    {"skill_raw": "ETL",                      "true_taxonomy_id": 52},
    {"skill_raw": "visão de produto",         "true_taxonomy_id": 14},
    {"skill_raw": "Product Sense",            "true_taxonomy_id": 14},
    {"skill_raw": "Alinhamento com o universo", "true_taxonomy_id": None},  # None
    {"skill_raw": "inteligência emocional",   "true_taxonomy_id": 4},
    {"skill_raw": "negociação",               "true_taxonomy_id": 5},

    {"skill_raw": "liderança",                "true_taxonomy_id": 25},
    {"skill_raw": "gestão de tempo",          "true_taxonomy_id": 10},
    {"skill_raw": "data analysis",            "true_taxonomy_id": 39},
    {"skill_raw": "deep learning",            "true_taxonomy_id": 41},
    {"skill_raw": "NLP",                      "true_taxonomy_id": 42},
    {"skill_raw": "docker",                   "true_taxonomy_id": 63},
    {"skill_raw": "kubernetes",               "true_taxonomy_id": 64},
    {"skill_raw": "CI/CD",                    "true_taxonomy_id": 65},
    {"skill_raw": "comunicação",              "true_taxonomy_id": 2},
    {"skill_raw": "resolução de problemas",   "true_taxonomy_id": 3},
    {"skill_raw": "tomada de decisão",        "true_taxonomy_id": 8},
    {"skill_raw": "adaptabilidade",           "true_taxonomy_id": 9},
    {"skill_raw": "colaboração",              "true_taxonomy_id": 13},
    {"skill_raw": "inovação",                 "true_taxonomy_id": 29},
    {"skill_raw": "gestão de projetos",       "true_taxonomy_id": 18},
    {"skill_raw": "planejamento estratégico", "true_taxonomy_id": 27},
    {"skill_raw": "data science",             "true_taxonomy_id": [39, 40]},
    {"skill_raw": "agilidade",                "true_taxonomy_id": 19},
    {"skill_raw": "feedback",                 "true_taxonomy_id": None},
    {"skill_raw": "astrologia",               "true_taxonomy_id": None},  # None
]

print(f'Total de instâncias para anotar: {len(ANNOTATIONS)}')
print(f'Já anotadas: {sum(1 for a in ANNOTATIONS if a["true_taxonomy_id"] is not None)}')
print(f'Pendentes:   {sum(1 for a in ANNOTATIONS if a["true_taxonomy_id"] is None)}')

Total de instâncias para anotar: 50
Já anotadas: 46
Pendentes:   4


## 4. Salvar anotações

In [16]:
annotations_df = pd.DataFrame(ANNOTATIONS)

# estatísticas rápidas
total      = len(annotations_df)
anotadas   = annotations_df['true_taxonomy_id'].notna().sum()
sem_match  = (annotations_df['true_taxonomy_id'].isna()).sum()
print(f'Total: {total} | Com match: {anotadas} | Sem match (None): {sem_match}')

# salva como CSV
annotations_df.to_csv(ANNOTATIONS_PATH, index=False)
print(f'\nAnotações salvas em: {ANNOTATIONS_PATH}')
display(annotations_df)

Total: 50 | Com match: 46 | Sem match (None): 4

Anotações salvas em: data/raw/manual_annotations.csv


,skill_raw,true_taxonomy_id
0,gestão de pessoas,1
1,People Management,1
2,SQL,32
3,sql avançado,32
4,machine learning,40
5,comunicação assertiva,2
6,python,31
7,PYTHON,31
8,análise de dados,39
9,resolução de conflitos,3


## 5. Avaliação com ground truth manual

Roda o matcher em todas as skills anotadas e calcula precision, recall e F1 para diferentes thresholds.

In [17]:
annotations_df = pd.read_csv(ANNOTATIONS_PATH)

gt_manual = annotations_df.dropna(subset=['true_taxonomy_id']).copy()
print(f'Ground truth manual: {len(gt_manual)} pares com match real')

all_skills_df = pd.DataFrame({
    'id':        range(len(annotations_df)),
    'skill_raw': annotations_df['skill_raw'].values,
})

curve_manual = precision_recall_curve_by_threshold(
    new_skills_df=all_skills_df,
    taxonomy_df=taxonomy_df,
    embedder=embedder,
    ground_truth=gt_manual,
)

print('\n=== Curva com anotações manuais ===')
display(curve_manual)

Ground truth manual: 46 pares com match real


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Sweeping thresholds: 100%|██████████| 13/13 [00:00<00:00, 92.39it/s]


=== Curva com anotações manuais ===


,threshold,precision,recall,f1,coverage
0,0.30,0.9333,0.9130,0.9231,0.96
1,0.35,0.9302,0.8696,0.8989,0.88
2,0.40,0.9524,0.8696,0.9091,0.86
3,0.45,0.9756,0.8696,0.9195,0.84
4,0.50,0.9750,0.8478,0.9070,0.82
5,0.55,0.9737,0.8043,0.8810,0.78
6,0.60,0.9706,0.7174,0.8250,0.68
7,0.65,0.9688,0.6739,0.7949,0.64
8,0.70,0.9583,0.5000,0.6571,0.48
9,0.75,0.9333,0.3043,0.4590,0.30


## 6. Comparação: curva manual vs. sintética

In [ ]:
from evaluator import generate_synthetic_ground_truth

# ground truth sintético com selective normalization (baseline do notebook 01)
gt_synthetic = generate_synthetic_ground_truth(taxonomy_df)
gt_skills_df = pd.DataFrame({'id': range(len(gt_synthetic)),
                              'skill_raw': gt_synthetic['skill_raw'].values})

curve_synthetic = precision_recall_curve_by_threshold(
    new_skills_df=gt_skills_df,
    taxonomy_df=taxonomy_df,
    embedder=embedder,
    ground_truth=gt_synthetic,
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, (curve, title) in zip(axes, [
    (curve_manual,   'Ground Truth Manual (selective, threshold 0.45)'),
    (curve_synthetic,'Ground Truth Sintético (selective, threshold 0.45)'),
]):
    ax.plot(curve['threshold'], curve['precision'], marker='o', label='Precision')
    ax.plot(curve['threshold'], curve['recall'],    marker='s', label='Recall')
    ax.plot(curve['threshold'], curve['f1'],        marker='^', label='F1', linewidth=2)
    ax.plot(curve['threshold'], curve['coverage'],  marker='D', label='Coverage', linestyle='--')
    ax.axvline(x=0.45, color='black', linestyle=':', linewidth=1.5, label='Threshold recomendado (0.45)')
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Score')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 1.05)

plt.suptitle('Precision / Recall / F1 / Coverage — GT Manual vs. GT Sintético', fontsize=13)
plt.tight_layout()
plt.savefig('data/output/curve_manual_vs_synthetic.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Distribuição de scores: modelo final vs. anterior

Comparação direta entre o modelo anterior (threshold 0.60, sem normalização) e o modelo final calibrado neste notebook (threshold 0.45, selective normalization). A linha tracejada marca o threshold de cada configuração.

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import matplotlib.pyplot as plt
import numpy as np
from matcher import SkillMatcher

results_old = pd.read_csv("data/output/result.csv")  # threshold 0.60

matcher_final = SkillMatcher(embedder=embedder, threshold=0.45)
results_final = matcher_final.match(skills_df, taxonomy_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
bins = np.linspace(0, 1, 26)

for ax, (results, thresh, title) in zip(axes, [
    (results_old,   0.60, "Modelo anterior  (threshold 0.60 | sem normalização)"),
    (results_final, 0.45, "Modelo final  (threshold 0.45 | selective normalization)"),
]):
    matched  = results[results["match_status"] == "matched"]["score"]
    no_match = results[results["match_status"] == "no_match"]["score"]
    ax.hist(matched,  bins=bins, alpha=0.75, color="#2196F3", label=f"matched  (n={len(matched)})")
    ax.hist(no_match, bins=bins, alpha=0.75, color="#F44336", label=f"no_match (n={len(no_match)})")
    ax.axvline(x=thresh, color="black", linestyle="--", linewidth=1.5, label=f"threshold ({thresh})")
    ax.set_xlabel("Cosine similarity score", fontsize=11)
    ax.set_ylabel("Count", fontsize=11)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("Distribuição de scores: modelo anterior vs. modelo final", fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig("data/output/score_distribution_final.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Anterior  — matched: {(results_old['match_status']=="matched").sum()}  no_match: {(results_old['match_status']=="no_match").sum()}")
print(f"Final     — matched: {(results_final['match_status']=="matched").sum()}  no_match: {(results_final['match_status']=="no_match").sum()}")


## 8. Métricas pontuais no threshold recomendado (0.45)

In [ ]:
row_manual    = curve_manual[curve_manual['threshold'] == 0.45].iloc[0]
row_synthetic = curve_synthetic[curve_synthetic['threshold'] == 0.45].iloc[0]

comparison = pd.DataFrame([
    {'ground_truth': 'Manual',    **row_manual.to_dict()},
    {'ground_truth': 'Sintético', **row_synthetic.to_dict()},
]).set_index('ground_truth')

print('=== Métricas no threshold recomendado (0.45) ===')
display(comparison.round(4))


## 9. Conclusão do notebook 02

### GT manual vs. GT sintético

O GT manual (46 pares anotados) revela que o modelo é significativamente mais preciso do que o GT sintético sugeria:

| Ground truth | Precision @0.45 | Recall @0.45 | F1 @0.45 |
|---|---|---|---|
| Sintético | 0.890 | 0.580 | 0.703 |
| **Manual** | **0.976** | **0.891** | **0.932** |

O GT sintético subestimava o F1 real porque suas variações artificiais (typos, siglas, traduções) geravam falsos positivos que não ocorrem na prática.

### Threshold 0.45: calibração definitiva

Com `selective` normalization e threshold 0.45:
- **Precision 0.976** — quase nenhum match errado retornado.
- **Recall 0.891** — cobre a grande maioria das skills mapeáveis.
- **F1 0.932** — máximo global dos experimentos.
- **Coverage 0.84** — 16% das skills vão para revisão manual.

O threshold anterior (0.60) sacrificava recall (0.717) sem ganho expressivo de precision (0.971 → 0.976 ao descer para 0.45).

### Distribuição de scores

O histograma final (modelo 0.45 + selective) mostra 27 matched / 3 no_match nas 30 skills brutas. Os 3 no_match restantes são skills sem correspondência real na taxonomia (`espiritualidade`, `Alinhamento com o universo`, `astrologia`) — rejeição correta.

### Multi-label e casos ambíguos

`cloud computing` e `data science` foram anotados com listas de IDs válidos. O evaluator os trata corretamente via `_valid_taxonomy_ids()`. Skills fora da taxonomia (`feedback`, `astrologia`) são consistentemente rejeitadas.

### Próximos passos

1. Atualizar threshold padrão de 0.60 → 0.45 no `SkillMatcher` e CLI.
2. Expandir GT manual com skills de MLOps e LLM frameworks.
3. Thresholds distintos por categoria (técnicas vs. comportamentais).
4. Reranking com LLM para scores em [0.40, 0.55].
